---
# Part 3 — Recommendation System

## Collaborative Filtering with ALS (Alternating Least Squares)

**Collaborative filtering** is the standard approach for recommendation systems. The core idea is that *users who agreed in the past tend to agree again in the future* — we fill in the missing entries of the user-item ratings matrix by leveraging patterns across all users.

Spark ML supports **model-based collaborative filtering**, where users and movies are each represented by a small set of **latent factors** (hidden features). The model learns these factors from observed ratings and uses them to predict unobserved ones.

### ALS Algorithm

Spark MLlib uses **Alternating Least Squares (ALS)** to factorize the ratings matrix **R** into two lower-rank matrices:

$$R \approx U \times V^T$$

- **U** — User latent factor matrix  
- **V** — Item (movie) latent factor matrix  

ALS alternates between fixing **U** and solving for **V**, then fixing **V** and solving for **U**, until convergence. This alternation makes the otherwise non-convex problem solvable as a series of least-squares problems, each of which is highly parallelizable on Spark.

 *See: [Spark MLlib Collaborative Filtering Docs](https://spark.apache.org/docs/latest/ml-collaborative-filtering.html)*

In [1]:
# give access to google colab to have access to your google drive, in order to be able to read the datasets
#from google.colab import drive
# drive.mount('/content/gdrive')
# google_drive_path = "/content/gdrive/MyDrive/Colab Notebooks/Data/DSC_511_PROJECT/"
from pathlib import Path
#import pandas as pd

# Notebook location (usually your project folder)
project_dir = Path.cwd()

# Data folder one level above project folder
data_dir = project_dir.parent / "data"   # change "data" if your folder name is different

print("Project dir:", project_dir)
print("Data dir:", data_dir)
print("Exists:", data_dir.exists())

# Optional: see files inside data folder
if data_dir.exists():
    for p in data_dir.iterdir():
        print(p.name)

Project dir: c:\Users\alexa\Documents\UCY files\Data Science\DSC511\DSC511 Project\DSC511-MovieLens-Project
Data dir: c:\Users\alexa\Documents\UCY files\Data Science\DSC511\DSC511 Project\data
Exists: True
links.csv
movies.csv
ratings.csv


In [11]:
%pip install pyspark

Note: you may need to restart the kernel to use updated packages.


In [5]:
!pip install numpy
!pip install pandas
!pip install matplotlib
!pip install seaborn

   ---------------------------------------- 0.0/12.3 MB ? eta -:--:--
   ----- ---------------------------------- 1.6/12.3 MB 14.7 MB/s eta 0:00:01
   ---------------------------------------- 12.3/12.3 MB 45.3 MB/s  0:00:00
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   -------- ------------------------------- 2.1/9.7 MB 16.0 MB/s eta 0:00:01
   ---------------------------------------- 9.7/9.7 MB 34.0 MB/s  0:00:00

   ---------------------------------------- 0/2 [tzdata]
   ---------------------------------------- 0/2 [tzdata]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------

In [ ]:
import os
os.environ["PYSPARK_SUBMIT_ARGS"] = "--driver-memory 20g pyspark-shell"

In [ ]:

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, explode
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.sql.functions import *
# from pyspark.sql.functions import col, from_unixtime, to_timestamp
#from pyspark.sql.functions import explode, split, col

#spark = SparkSession.builder.appName("DSC_511_PROJECT").master("local[*]").getOrCreate()
spark = SparkSession.builder \
    .appName("DSC_511_PROJECT") \
    .master("local[*]") \
    .config("spark.driver.memory", "20g") \
    .config("spark.driver.maxResultSize", "8g") \
    .config("spark.sql.shuffle.partitions", "200") \
    .getOrCreate()
# spark session for google colab
# spark = SparkSession.builder.appName("DSC_511_PROJECT").master("local[*]").config("spark.driver.memory", "12g").config("spark.driver.maxResultSize", "4g").config("spark.sql.shuffle.partitions", "200").config("spark.default.parallelism", "200").getOrCreate()

In [4]:
# Read the ratings and movies data into Spark DataFrames
ratings_df = spark.read.csv(str(data_dir / 'ratings.csv'), header=True, inferSchema=True)
movies_df = spark.read.csv(str(data_dir / 'movies.csv'), header=True, inferSchema=True)

In [5]:
# Pre processing steps of main analysis
from itertools import count


ratings_clean = ratings_df.withColumn("ts", to_timestamp(from_unixtime(col("timestamp"))))
# merging the dataframes
movies_ratings_merged = ratings_clean.join(movies_df, on="movieId", how="inner")
# removing the row with the genre that is a title
bad_id = movies_df.filter(col("genres").contains("We're Comin'")).select("movieId").first()[0]
#print("Corrupted movieId:", bad_id)
# Drop from all 3 dataframes
movies_df             = movies_df.filter(col("movieId") != bad_id)
ratings_df            = ratings_df.filter(col("movieId") != bad_id)
movies_ratings_merged = movies_ratings_merged.filter(col("movieId") != bad_id)

# other RDD/DataFrames created that might be useful for the analysis
#rating_dist = movies_ratings_merged.groupBy("rating").count().orderBy("rating").toPandas()
#movies_with_year = movies_df.withColumn("release_year",regexp_extract(col("title"), r"\((\d{4})\)", 1))
# top 10 most active users
# Get top 10 users by number of ratings given
#top10_users = movies_ratings_merged.groupBy("userId").agg(count("rating").alias("num_ratings"),round(avg("rating"), 2).alias("avg_rating")).orderBy("num_ratings", ascending=False).limit(10)

---
## Data Split Strategy

We use a **60 / 20 / 20** split:

| Split | Fraction | Purpose |
|---|---|---|
| `training_data` | 60% | Train both the baseline and the best model |
| `val_data` | 20% | Evaluate and **compare** baseline vs best model |
| `test_data` | 20% | Final evaluation — touched **once**, at the very end |

This avoids **data leakage**: the test set is completely held out during all model selection decisions. The validation set is used to compare models fairly after tuning.

In [6]:
# 60 / 20 / 20 split
(training_data, val_data, test_data) = ratings_df.randomSplit([0.6, 0.2, 0.2], seed=48)

training_data.persist()
val_data.persist()
# test_data is NOT persisted — we touch it only once at the very end

print(f"Training : {training_data.count():,}")
print(f"Validation: {val_data.count():,}")
print(f"Test     : {test_data.count():,}")

Training : 20,299,342
Validation: 6,764,910
Test     : 6,767,909


---
## Baseline ALS Model

Before tuning, we train a baseline ALS model with default parameters and evaluate it on the **validation set**.
This gives us a reference point to measure how much the hyperparameter tuning improves performance.

### Key Parameters

| Parameter | Value | Meaning |
|---|---|---|
| `rank` | 10 | Number of latent factors (size of U and V matrices) |
| `maxIter` | 10 | ALS iterations |
| `regParam` | 0.01 | Regularization — prevents overfitting |
| `coldStartStrategy` | `"drop"` | Drop users/items in validation set not seen during training, to avoid NaN predictions breaking RMSE |
| `seed` | 48 | Fixed seed for reproducibility — ALS initializes U and V randomly |

In [8]:
# Baseline ALS model
als = ALS(
    rank=10,
    maxIter=10,
    regParam=0.01,
    userCol="userId",
    itemCol="movieId",
    ratingCol="rating",
    coldStartStrategy="drop",
    seed=48
)

model = als.fit(training_data)
print("Baseline model trained.")

Baseline model trained.


### Baseline Model Evaluation

We evaluate using **Root Mean Squared Error (RMSE)** and **Mean Absolute Error (MAE)** on the **validation set**.

$$RMSE = \sqrt{\frac{1}{n}\sum_{i=1}^{n}(\hat{r}_i - r_i)^2}$$

Since MovieLens ratings are on a 0.5–5.0 scale, a RMSE in the range of 0.8–1.0 is generally considered good.

In [9]:
# Evaluate baseline on validation set
baseline_preds = model.transform(val_data)

cv_evaluator = RegressionEvaluator(metricName="rmse", labelCol="rating", predictionCol="prediction")
mae_evaluator = RegressionEvaluator(metricName="mae",  labelCol="rating", predictionCol="prediction")

baseline_rmse = cv_evaluator.evaluate(baseline_preds)
baseline_mae  = mae_evaluator.evaluate(baseline_preds)

print(f"Baseline RMSE (validation) : {baseline_rmse:.4f}")
print(f"Baseline MAE  (validation) : {baseline_mae:.4f}")

Baseline RMSE (validation) : 0.8456
Baseline MAE  (validation) : 0.6344


---
## Hyperparameter Tuning — Cross Validation on a 10% Sample of Training Data

Training the full ALS model on ~20M ratings (60% of 33M) is computationally expensive, making exhaustive hyperparameter search on the complete training set impractical. A standard and widely accepted approach in Big Data is to **tune hyperparameters on a representative sample of the training data**, then retrain the best configuration on the full training set.

We sample **20% of `training_data`** (~4M ratings). Crucially, this sample comes **only from the training set** — the validation and test sets are never touched during this process, ensuring no data leakage.

### Search Space

| Parameter | Values | Description |
|---|---|---|
| `rank` | 10, 15 | Number of latent factors |
| `regParam` | 0.01, 0.1 | Regularization strength |
| `maxIter` | 5, 10, 15 | Number of ALS iterations |

This gives **2 × 2 × 2 = 8 configurations × 5 folds = 40 ALS fits** total.
The best configuration is selected based on **mean RMSE across all 5 folds**.

In [10]:
#from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
import pandas as pd

# ── 20% sample from training_data only — val/test never touched ────
train_sample = training_data.sample(fraction=0.2, seed=48)
train_sample.persist()
print(f"CV sample size: {train_sample.count():,}")

# ── ALS base estimator (no fixed hyperparams — grid fills them in) ─
als_tuning = ALS(
    userCol="userId",
    itemCol="movieId",
    ratingCol="rating",
    coldStartStrategy="drop",
    seed=48
)

# ── Parameter grid: 2 × 2 × 3 = 12 combinations ────────────────────
param_grid = (
    ParamGridBuilder()
    .addGrid(als_tuning.rank,     [10, 15])
    .addGrid(als_tuning.regParam, [0.01, 0.1])
    .addGrid(als_tuning.maxIter,  [5, 10, 15])
    .build()
)

# ── 5-fold Cross Validation on the sample — selection based on RMSE 
cv = CrossValidator(
    estimator=als_tuning,
    estimatorParamMaps=param_grid,
    evaluator=cv_evaluator,   # RMSE evaluator defined above
    numFolds=5,
    seed=48
)

cv_model = cv.fit(train_sample)
train_sample.unpersist()

CV sample size: 4,061,836


DataFrame[userId: int, movieId: int, rating: double, timestamp: int]

In [11]:
# ── Results table: mean RMSE per configuration ─────────────────────
results = []
for params, avg_rmse in zip(cv_model.getEstimatorParamMaps(), cv_model.avgMetrics):
    results.append({
        "rank"     : params[als_tuning.rank],
        "regParam" : params[als_tuning.regParam],
        "maxIter"  : params[als_tuning.maxIter],
        "mean_rmse": float(avg_rmse)
    })

results_df = pd.DataFrame(results).sort_values("mean_rmse")
print("\nCross-Validation Results (sorted by mean RMSE):")
print(results_df.to_string(index=False))

# ── Best configuration ─────────────────────────────────────────────
best_rank      = cv_model.bestModel.rank
best_reg_param = cv_model.bestModel._java_obj.parent().getRegParam()
best_max_iter  = cv_model.bestModel._java_obj.parent().getMaxIter()
print(f"\n Best rank     : {best_rank}")
print(f" Best regParam : {best_reg_param}")
print(f" Best maxIter  : {best_max_iter}")
print(f" Best mean RMSE (CV on sample): {results_df.iloc[0]['mean_rmse']}")


Cross-Validation Results (sorted by mean RMSE):
 rank  regParam  maxIter  mean_rmse
   10      0.10       15   0.910447
   15      0.10       15   0.913633
   10      0.10       10   0.917613
   15      0.10       10   0.920865
   10      0.10        5   0.932251
   15      0.10        5   0.935875
   10      0.01        5   1.118572
   10      0.01       10   1.145541
   10      0.01       15   1.173665
   15      0.01        5   1.188082
   15      0.01       10   1.211942
   15      0.01       15   1.239764

 Best rank     : 10
 Best regParam : 0.1
 Best maxIter  : 15
 Best mean RMSE (CV on sample): 0.9104471716335132


---
## Retrain Best Configuration on Full Training Data

The cross-validation selected the best hyperparameters using a 10% sample of the training set.
We now retrain a new ALS model using those best parameters on the **full `training_data`** (100%, ~20M ratings).

We then evaluate both the **baseline** and the **best model** on the **validation set** to measure the improvement brought by tuning.

In [12]:
# Retrain with best params on full training_data
print("Retraining best configuration on full training data...")
als_best = ALS(
    rank=best_rank,
    regParam=best_reg_param,
    maxIter=best_max_iter,
    userCol="userId",
    itemCol="movieId",
    ratingCol="rating",
    coldStartStrategy="drop",
    seed=48
)

best_model = als_best.fit(training_data)
print("Best model trained.")

# ── Compare baseline vs best model on validation set ──────────────
best_val_preds = best_model.transform(val_data)
best_val_rmse  = cv_evaluator.evaluate(best_val_preds)
best_val_mae   = mae_evaluator.evaluate(best_val_preds)

print("\n── Model Comparison on Validation Set ──────────────────────")
print(f"{'Model':<20} {'RMSE':>8} {'MAE':>8}")
print("-" * 40)
print(f"{'Baseline':<20} {baseline_rmse:>8.4f} {baseline_mae:>8.4f}")
print(f"{'Best (tuned)':<20} {best_val_rmse:>8.4f} {best_val_mae:>8.4f}")
print(f"{'Improvement':<20} {baseline_rmse - best_val_rmse:>8.4f} {baseline_mae - best_val_mae:>8.4f}")

Retraining best configuration on full training data...
Best model trained.

── Model Comparison on Validation Set ──────────────────────
Model                    RMSE      MAE
----------------------------------------
Baseline               0.8456   0.6344
Best (tuned)           0.8116   0.6241
Improvement            0.0340   0.0102


---
## Final Evaluation on Test Set

The test set has been completely held out throughout the entire process — it was never used for training, validation, or model selection. We now evaluate the best model on the test set **once** to report the final, unbiased performance.

In [13]:
# Final evaluation — test set touched for the first and only time
final_preds = best_model.transform(test_data)

final_rmse = cv_evaluator.evaluate(final_preds)
final_mae  = mae_evaluator.evaluate(final_preds)

print("── Final Model Performance on Test Set ─────────────────────")
print(f"RMSE : {final_rmse:.4f}")
print(f"MAE  : {final_mae:.4f}")

# Free validation data — no longer needed
val_data.unpersist()
training_data.unpersist()

── Final Model Performance on Test Set ─────────────────────
RMSE : 0.8120
MAE  : 0.6243


DataFrame[userId: int, movieId: int, rating: double, timestamp: int]

---
## Generate Recommendations

Using the best trained model, we generate the top-10 movie recommendations for every user.

`recommendForAllUsers(N)` uses the learned latent factors to predict ratings for every movie each user has not yet seen and returns the top-N per user. The result is a nested array column which we flatten using `explode()` — exactly as done in the course lab.

In [14]:
from pyspark.sql.functions import col, explode

# Top 10 movie recommendations for every user
user_recs = best_model.recommendForAllUsers(10)

# Explode the recommendations array into one row per (user, movie)
user_recs = user_recs.select(col("userId"), explode(col("recommendations")).alias("recommendations"))

# Unpack the struct column into separate movieId and rating columns
user_recs = user_recs.selectExpr(
    "userId",
    "recommendations.movieId as movieId",
    "recommendations.rating  as rating"
)

user_recs.show(10, truncate=False)

+------+-------+---------+
|userId|movieId|rating   |
+------+-------+---------+
|31    |222368 |5.005954 |
|31    |270306 |4.389038 |
|31    |240070 |4.389038 |
|31    |194434 |4.3725805|
|31    |226196 |4.3099103|
|31    |136892 |4.2802167|
|31    |169908 |4.2607226|
|31    |275847 |4.0912952|
|31    |205277 |4.0443745|
|31    |231289 |4.0433474|
+------+-------+---------+
only showing top 10 rows


### Recommendations for a Specific User

We pick a specific user, show what they have already rated, and then show the top-10 recommendations joined with `movies_df` to display human-readable titles and genres.

In [15]:
TARGET_USER = 4

# Movies already rated by the user — use movies_ratings_merged (no extra join needed)
print(f"Movies already rated by User {TARGET_USER}:")
(
    movies_ratings_merged
    .filter(col("userId") == TARGET_USER)
    .select("title", "genres", "rating")
    .orderBy(desc("rating"))
    .show(20, truncate=False)
)

# Top 10 recommendations — user_recs filtered to 10 rows, broadcast join with movies_df
print(f"Top 10 recommendations for User {TARGET_USER}:")
(
    user_recs
    .filter(col("userId") == TARGET_USER)
    .join(movies_df, on="movieId", how="inner")
    .select("title", "genres", "rating")
    .orderBy(desc("rating"))
    .show(truncate=False)
)

Movies already rated by User 4:
+----------------------------------------------------+---------------------------------------------------+------+
|title                                               |genres                                             |rating|
+----------------------------------------------------+---------------------------------------------------+------+
|Star Wars: Episode IV - A New Hope (1977)           |Action|Adventure|Sci-Fi                            |5.0   |
|Shawshank Redemption, The (1994)                    |Crime|Drama                                        |5.0   |
|Forrest Gump (1994)                                 |Comedy|Drama|Romance|War                           |5.0   |
|Life Is Beautiful (La Vita è bella) (1997)          |Comedy|Drama|Romance|War                           |5.0   |
|Spirited Away (Sen to Chihiro no kamikakushi) (2001)|Adventure|Animation|Fantasy                        |5.0   |
|Pianist, The (2002)                                 |Dr

In [16]:
# Generate top 10 user recommendations for each movie
movieRecs = best_model.recommendForAllItems(10)

# convert the recommendations to multiple rows per movie with one recommendation in each row
movieRecs = movieRecs.selectExpr("movieId", "explode(recommendations) as recommendations")
# convert the recommendations column from {userId, rating} to tow columns userId  and rating
movieRecs = movieRecs.selectExpr("movieId", "recommendations.userId as userId",
                                    "recommendations.rating as rating")

# Show the recommendations for a specific movie
movie_id = 4000

movie_rec = (
    movieRecs
    .filter(movieRecs.movieId == movie_id)
    .join(movies_df, on="movieId", how="inner")
    .select("movieId", "title", "genres", "userId", "rating")
    .orderBy(desc("rating"))
)

movie_title = movies_df.filter(col("movieId") == movie_id).select("title").first()[0]
print(f"Top 10 users most likely to enjoy: '{movie_title}'")
movie_rec.show(truncate=False)


Top 10 users most likely to enjoy: 'Bounty, The (1984)'
+-------+------------------+---------------+------+---------+
|movieId|title             |genres         |userId|rating   |
+-------+------------------+---------------+------+---------+
|4000   |Bounty, The (1984)|Adventure|Drama|322890|5.7422047|
|4000   |Bounty, The (1984)|Adventure|Drama|112496|5.448017 |
|4000   |Bounty, The (1984)|Adventure|Drama|132881|5.168941 |
|4000   |Bounty, The (1984)|Adventure|Drama|30789 |5.1527987|
|4000   |Bounty, The (1984)|Adventure|Drama|25841 |5.095664 |
|4000   |Bounty, The (1984)|Adventure|Drama|319170|5.0789766|
|4000   |Bounty, The (1984)|Adventure|Drama|162165|5.07125  |
|4000   |Bounty, The (1984)|Adventure|Drama|79911 |5.0560737|
|4000   |Bounty, The (1984)|Adventure|Drama|58782 |5.052303 |
|4000   |Bounty, The (1984)|Adventure|Drama|136723|5.0522118|
+-------+------------------+---------------+------+---------+



In [29]:
best_params = {
    "rank": best_rank,
    "regParam": best_reg_param,
    "maxIter": best_max_iter
}
print("\nBest Hyperparameters:")
for param, value in best_params.items():
    print(f"{param}: {value}")


Best Hyperparameters:
rank: 10
regParam: 0.1
maxIter: 15
